In [32]:
import pandas as pd
from joblib import load
from sklearn.metrics import f1_score,confusion_matrix
import numpy as np

In [33]:
oxide = pd.read_excel(r"D:/Research/Mineral identifier ann IISER Mohali/new random comp generator/GEOROC data/processed minerals/oxide_data.xlsx", sheet_name="Sheet1",index_col=0,header=0)

def normalize(data,n=100):
    s = n/data.sum(axis=1)
    data_formatted = data.mul(s,axis=0)
    return(data_formatted)
    
def wt_to_mol(data, oxide = oxide):
    data[data<=2] = 0
    if 'Total' in data.columns:
        data.pop('Total')
    oxlist1 = data.columns
    oxide = oxide.T[oxlist1].iloc[0, :]
    data = normalize(data,100)
    data_f = data.div(oxide,axis=1)
    data_f = normalize(data_f)
    return(data_f.round(2))

# file = r"D:/Research/Mineral identifier ann IISER Mohali/new random comp generator/GEOROC data/processed minerals/molar tables/new data/compiled9_2.xlsx"
# file_test = r"D:/Research/Mineral identifier ann IISER Mohali/new random comp generator/External test data/Mineral data combined_RSandSS_mol1.xlsx"
# file_test = r"D:\Research\Mineral identifier ann IISER Mohali\new random comp generator\External test data\Mineral data combined_DHZ,PSandSS.xlsx"
# file_test = r"D:\Research\Mineral identifier ann IISER Mohali\new random comp generator\External test data\test_data_deerhowiezussmann.xlsx"
file_test = r"D:\Research\Mineral identifier ann IISER Mohali\ms and reference papers\MinNet ver 4 for submitted to AmMin\Revision 1\AmMin revision 1\Sir modified files\External dataset 2\Processed\name shortened\Combined_PantSir.xlsx"

# data = pd.read_excel(file,header=0,index_col=0)
data_test = pd.read_excel(file_test,header=0,index_col=0)
m = len(data_test['Mineral'].unique())
acc_table = np.zeros((m+1,5))

data_test

,SiO2,MgO,FeO,TiO2,Al2O3,MnO,CaO,Na2O,K2O,Cr2O3,...,Er2O3,Yb2O3,V2O3,CoO,ZnO,ZrO2,CuO,Total,Mineral,Source
1,40.79,48.85,10.16,0.000,0.00,0.000,0.000,0.000,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,99.800,Ol,Batanova et al 2019
2,40.47,48.68,10.20,0.000,0.00,0.000,0.000,0.000,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,99.350,Ol,Batanova et al 2019
3,40.96,48.83,10.16,0.000,0.00,0.000,0.000,0.000,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,99.950,Ol,Batanova et al 2019
4,55.77,6.37,0.02,0.010,23.62,0.000,0.000,0.100,11.23,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,97.120,Ms,Biino and Gröning - 1998
5,55.65,6.29,0.01,0.000,23.67,0.000,0.000,0.080,11.17,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,96.870,Ms,Biino and Gröning - 1998
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1422,48.50,0.00,4.40,0.027,26.50,1.327,0.008,0.324,10.20,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,91.286,Ms,Xu et al 2023
1423,47.40,1.26,4.07,0.363,32.50,0.221,0.017,0.617,9.57,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,96.018,Ms,Xu et al 2023
1424,45.60,0.52,4.76,0.357,32.00,0.694,0.011,0.699,9.90,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,94.541,Ms,Xu et al 2023
1425,47.80,0.00,0.93,0.067,36.60,0.349,0.002,0.327,10.31,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,96.385,Ms,Xu et al 2023


In [34]:
if 'Mineral' in data_test.columns:
    minerals = data_test.pop("Mineral")
if 'Source' in data_test.columns:
    data_test.pop("Source")

# if 'Fe2O3' in data_test.columns:
#     data_test['FeO'] = data_test['FeO'] + (0.8998 * data_test['Fe2O3'])
#     data_test.pop('Fe2O3')

metrics = 'weighted'
if 'F' in data_test.columns:
    data_test.pop('F')
if 'Cl' in data_test.columns:
    data_test.pop('Cl')

data_test = normalize(data_test)

if 'Total' not in data_test.columns:
    data_test['Total'] = data_test.iloc[:,:-2].sum(axis=1)
if 'P2O5' not in data_test.columns:
    data_test['P2O5']=0
if 'CO2' not in data_test.columns:
    data_test['CO2']=0
    data_test.loc[minerals=='Cb','CO2'] = data_test['Total'] - data_test[['FeO','MnO','MgO','CaO']].sum(axis=1)

In [35]:
data_test.fillna(0,inplace=True)

In [36]:
data_test = wt_to_mol(data_test,oxide)
if 'SrO' in data_test.columns:
    data_test['CaO'] = data_test['CaO'] + data_test['SrO']
    data_test.pop('SrO')
if 'BaO' in data_test.columns:
    data_test['CaO'] = data_test['CaO'] + data_test['BaO']
    data_test.pop('BaO')
if 'Cr2O3' in data_test.columns:
    data_test['Al2O3'] = data_test['Al2O3'] + data_test['Cr2O3']
    data_test.pop('Cr2O3')
if 'ZnO' in data_test.columns:
    data_test['FeO'] = data_test['FeO'] + data_test['ZnO']
    data_test.pop('ZnO')
if 'CoO' in data_test.columns:
    data_test['FeO'] = data_test['FeO'] + data_test['CoO']
    data_test.pop('CoO')

In [37]:
data_test_original = data_test.copy()

# Raw Input (C1)

In [38]:
model = load("RF_C1.mdl")
# scaler = load("Scaler_C1.scl")
labeler = load("Labeler_C1.lbl")

data_test = data_test_original.copy()
data_test1 = data_test.copy()
# data_test1['M'] = data_test1[["FeO","MnO","MgO"]].sum(axis=1)
data_test1 = data_test1[['SiO2','TiO2','Al2O3','FeO','MnO','MgO','CaO','Na2O','K2O','P2O5','CO2']]

# m = data_test1[["FeO","MnO","MgO"]]sum(axis=1)

# data_test_scaled = scaler.transform(data_test1)

data_test_scaled =data_test1.copy()
test_target = labeler.transform(minerals)
pred_label = model.predict(data_test_scaled)
a = round(f1_score(test_target, pred_label, average=metrics,zero_division=0),4)*100
acc_table[0,0] = a
print("Overall: "+str(a))

labels = labeler.transform(minerals.unique())
labels1 = labeler.inverse_transform(labels)

for n,i in enumerate(labels1):
    # data_test2 = scaler.transform(data_test1.loc[minerals==i,:])
    data_test2 = data_test1.loc[minerals==i,:].copy()

    minerals2 = labeler.transform(minerals.loc[minerals == i])
    pred_label2 = model.predict(data_test2)
    a = round(f1_score(minerals2, pred_label2, average=metrics,zero_division=0),4)*100
    acc_table[n+1,0] = a
    print(labels1[n]+": "+str(round(a,2)))

data_mismatch = data_test.loc[test_target != pred_label,:]
minerals_mismatch = minerals.loc[test_target != pred_label]
data_mismatch = pd.concat([data_mismatch,minerals_mismatch],axis=1)
data_mismatch

C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomFo

Overall: 97.18
Ol: 100.0
Ms: 98.7
Grt: 83.44
Amp: 98.23
Px: 95.71
Spl: 100.0
Fsp: 99.56
Bt: 100.0
Ttn: 100.0
Ap: 100.0


C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


,SiO2,MgO,FeO,TiO2,Al2O3,MnO,CaO,Na2O,K2O,P2O5,...,Sm2O3,Gd2O3,Dy2O3,Er2O3,Yb2O3,V2O3,ZrO2,CuO,CO2,Mineral
18,45.840504,0.0,28.607461,0.0,13.61215,0.0,11.939885,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Grt
66,51.715743,14.21163,14.840854,0.0,9.981523,0.0,9.25025,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
68,50.033745,22.631168,7.420503,0.0,7.967547,0.0,11.947036,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
834,51.060666,6.660744,0.0,0.0,19.671164,0.0,22.607426,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Fsp
867,42.048091,20.171241,10.876221,3.581093,9.354042,0.0,13.969312,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Amp
869,59.307416,0.0,27.103482,0.0,0.0,0.0,0.0,13.589102,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Amp
871,44.359125,21.170889,12.046923,0.0,9.609952,0.0,12.813111,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Amp
872,42.645517,27.608024,3.742378,0.0,11.013908,0.0,14.990172,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Amp
922,53.51399,26.335675,5.756234,0.0,0.0,0.0,14.394102,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
923,57.865942,13.418328,0.0,0.0,7.848912,0.0,14.68308,6.183738,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px


# C2

In [39]:
model = load("RF_C2.mdl")
# scaler = load("Scaler_C2.scl")
labeler = load("Labeler_C2.lbl")

data_test = data_test_original.copy()

data_test1 = data_test.copy()
data_test1['M'] = data_test1[["FeO","MnO","MgO"]].sum(axis=1)
data_test1 = data_test1[['SiO2','TiO2','Al2O3','M','CaO','Na2O','K2O','P2O5','CO2']]

# m = data_test1[["FeO","MnO","MgO"]]sum(axis=1)

# data_test_scaled = scaler.transform(data_test1)
data_test_scaled =data_test1.copy()
test_target = labeler.transform(minerals)
pred_label = model.predict(data_test_scaled)
a = round(f1_score(test_target, pred_label, average=metrics,zero_division=0),4)*100
acc_table[0,1] = a
print("Overall: "+str(a))

labels = labeler.transform(minerals.unique())
labels1 = labeler.inverse_transform(labels)

for n,i in enumerate(labels1):
    # data_test2 = scaler.transform(data_test1.loc[minerals==i,:])
    data_test2 = data_test1.loc[minerals==i,:].copy()

    minerals2 = labeler.transform(minerals.loc[minerals == i])
    pred_label2 = model.predict(data_test2)
    a = round(f1_score(minerals2, pred_label2, average=metrics,zero_division=0),4)*100
    acc_table[n+1,1] = a
    print(labels1[n]+": "+str(round(a,2)))

data_mismatch = data_test.loc[test_target != pred_label,:]
minerals_mismatch = minerals.loc[test_target != pred_label]
data_mismatch = pd.concat([data_mismatch,minerals_mismatch],axis=1)
# data_mismatch['Pred'] = labeler.inverse_transform(pred_label)
data_mismatch

C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomFo

Overall: 97.04
Ol: 100.0
Ms: 94.59
Grt: 84.85
Amp: 97.32
Px: 96.45
Spl: 100.0
Fsp: 100.0
Bt: 99.82
Ttn: 100.0
Ap: 100.0


C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


,SiO2,MgO,FeO,TiO2,Al2O3,MnO,CaO,Na2O,K2O,P2O5,...,Sm2O3,Gd2O3,Dy2O3,Er2O3,Yb2O3,V2O3,ZrO2,CuO,CO2,Mineral
4,64.587033,10.997559,0.0,0.0,16.119588,0.0,0.0,0.0,8.295821,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Ms
5,64.633747,10.890729,0.0,0.0,16.200252,0.0,0.0,0.0,8.275272,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Ms
17,45.482901,0.0,29.810474,0.0,13.607599,0.0,11.099025,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Grt
18,45.840504,0.0,28.607461,0.0,13.61215,0.0,11.939885,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Grt
66,51.715743,14.21163,14.840854,0.0,9.981523,0.0,9.25025,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
68,50.033745,22.631168,7.420503,0.0,7.967547,0.0,11.947036,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
75,48.231447,25.375501,10.503592,0.0,5.554196,0.0,10.335263,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Amp
717,45.416186,35.67116,6.281221,0.0,12.631434,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Grt
867,42.048091,20.171241,10.876221,3.581093,9.354042,0.0,13.969312,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Amp
871,44.359125,21.170889,12.046923,0.0,9.609952,0.0,12.813111,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Amp


# C3

In [40]:
model = load("RF_C3.mdl")
# scaler = load("Scaler_C3.scl")
labeler = load("Labeler_C3.lbl")

data_test = data_test_original.copy()

data_test1 = data_test.copy()
data_test1['M'] = data_test1[["FeO","MnO","MgO"]].sum(axis=1)
data_test1['A'] = data_test1[["Na2O","K2O"]].sum(axis=1)
data_test1 = data_test1[['SiO2','TiO2','Al2O3','M','CaO','A','P2O5','CO2']]

# m = data_test1[["FeO","MnO","MgO"]]sum(axis=1)

# data_test_scaled = scaler.transform(data_test1)
data_test_scaled =data_test1.copy()
test_target = labeler.transform(minerals)
pred_label = model.predict(data_test_scaled)
a = round(f1_score(test_target, pred_label, average=metrics,zero_division=0),4)*100
acc_table[0,2] = a
print("Overall: "+str(a))

labels = labeler.transform(minerals.unique())
labels1 = labeler.inverse_transform(labels)

for n,i in enumerate(labels1):
    # data_test2 = scaler.transform(data_test1.loc[minerals==i,:])
    data_test2 = data_test1.loc[minerals==i,:].copy()

    minerals2 = labeler.transform(minerals.loc[minerals == i])
    pred_label2 = model.predict(data_test2)
    a = round(f1_score(minerals2, pred_label2, average=metrics,zero_division=0),4)*100
    acc_table[n+1,2] = a
    print(labels1[n]+": "+str(round(a,2)))

data_mismatch = data_test.loc[test_target != pred_label,:]
minerals_mismatch = minerals.loc[test_target != pred_label]
data_mismatch = pd.concat([data_mismatch,minerals_mismatch],axis=1)
data_mismatch

C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomFo

Overall: 98.09
Ol: 100.0
Ms: 93.15
Grt: 93.85
Amp: 97.78
Px: 97.18
Spl: 100.0
Fsp: 100.0
Bt: 99.65
Ttn: 100.0
Ap: 100.0


C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


,SiO2,MgO,FeO,TiO2,Al2O3,MnO,CaO,Na2O,K2O,P2O5,...,Sm2O3,Gd2O3,Dy2O3,Er2O3,Yb2O3,V2O3,ZrO2,CuO,CO2,Mineral
4,64.587033,10.997559,0.0,0.0,16.119588,0.0,0.0,0.0,8.295821,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Ms
5,64.633747,10.890729,0.0,0.0,16.200252,0.0,0.0,0.0,8.275272,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Ms
66,51.715743,14.21163,14.840854,0.0,9.981523,0.0,9.25025,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
68,50.033745,22.631168,7.420503,0.0,7.967547,0.0,11.947036,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
75,48.231447,25.375501,10.503592,0.0,5.554196,0.0,10.335263,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Amp
860,49.687519,0.0,24.52711,0.0,17.817903,0.0,0.0,0.0,7.967468,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Bt
867,42.048091,20.171241,10.876221,3.581093,9.354042,0.0,13.969312,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Amp
872,42.645517,27.608024,3.742378,0.0,11.013908,0.0,14.990172,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Amp
876,56.035399,31.735211,4.55848,0.0,0.0,0.0,7.67091,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Amp
889,55.617532,31.635973,4.718555,0.0,0.0,0.0,8.02794,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Amp


# C4

In [41]:
model = load("RF_C4.mdl")
# scaler = load("Scaler_C4.scl")
labeler = load("Labeler_C4.lbl")

data_test = data_test_original.copy()

data_test1 = data_test.copy()
data_test1['M'] = data_test1[["FeO","MnO","MgO","CaO"]].sum(axis=1)
data_test1['A'] = data_test1[["CaO","Na2O","K2O"]].sum(axis=1)
data_test1 = data_test1[['SiO2','TiO2','Al2O3','M','A','P2O5','CO2']]

# m = data_test1[["FeO","MnO","MgO"]]sum(axis=1)

# data_test_scaled = scaler.transform(data_test1)
data_test_scaled =data_test1.copy()
test_target = labeler.transform(minerals)
pred_label = model.predict(data_test_scaled)
a = round(f1_score(test_target, pred_label, average=metrics,zero_division=0),4)*100
acc_table[0,3] = a
print("Overall: "+str(a))

labels = labeler.transform(minerals.unique())
labels1 = labeler.inverse_transform(labels)

for n,i in enumerate(labels1):
    # data_test2 = scaler.transform(data_test1.loc[minerals==i,:])
    data_test2 = data_test1.loc[minerals==i,:].copy()
    minerals2 = labeler.transform(minerals.loc[minerals == i])
    pred_label2 = model.predict(data_test2)
    a = round(f1_score(minerals2, pred_label2, average=metrics,zero_division=0),4)*100
    acc_table[n+1,3] = a
    print(labels1[n]+": "+str(round(a,2)))

data_mismatch = data_test.loc[test_target != pred_label,:]
minerals_mismatch = minerals.loc[test_target != pred_label]
data_mismatch = pd.concat([data_mismatch,minerals_mismatch],axis=1)
data_mismatch

C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomFo

Overall: 92.05
Ol: 100.0
Ms: 100.0
Grt: 62.32
Amp: 99.12
Px: 96.45
Spl: 100.0
Fsp: 71.51
Bt: 99.47
Ttn: 100.0
Ap: 100.0


C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


,SiO2,MgO,FeO,TiO2,Al2O3,MnO,CaO,Na2O,K2O,P2O5,...,Sm2O3,Gd2O3,Dy2O3,Er2O3,Yb2O3,V2O3,ZrO2,CuO,CO2,Mineral
17,45.482901,0.0,29.810474,0.0,13.607599,0.0,11.099025,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Grt
18,45.840504,0.0,28.607461,0.0,13.61215,0.0,11.939885,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Grt
66,51.715743,14.21163,14.840854,0.0,9.981523,0.0,9.25025,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
68,50.033745,22.631168,7.420503,0.0,7.967547,0.0,11.947036,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
719,42.957666,29.502187,9.853502,0.0,12.519364,0.0,5.167281,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Grt
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1411,48.409745,0.0,36.098709,0.0,15.491547,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Grt
1412,48.543643,0.0,36.35922,0.0,15.097137,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Grt
1413,48.158626,0.0,36.571271,0.0,15.270103,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Grt
1414,48.057343,0.0,36.728215,0.0,15.214442,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Grt


In [42]:
data_mismatch[data_mismatch['Mineral']=="Cb"]

,SiO2,MgO,FeO,TiO2,Al2O3,MnO,CaO,Na2O,K2O,P2O5,...,Sm2O3,Gd2O3,Dy2O3,Er2O3,Yb2O3,V2O3,ZrO2,CuO,CO2,Mineral


# C5

In [43]:
model = load("RF_C5_5_components.mdl")
# scaler = load("Scaler_C5_5_components.scl")
labeler = load("Labeler_C5_5_components.lbl")
pc = load("PCA_C5_5_components.pc")

data_test = data_test_original.copy()

data_test1 = data_test.copy()
data_test1 = data_test1[['SiO2','TiO2','Al2O3','FeO','MnO','MgO','CaO','Na2O','K2O','P2O5','CO2']]
# m = data_test1[["FeO","MnO","MgO"]]sum(axis=1)

# data_test_scaled = scaler.transform(data_test1)
data_test_scaled =data_test1.copy()
data_test_scaled = pc.transform(data_test_scaled)
test_target = labeler.transform(minerals)
pred_label = model.predict(data_test_scaled)
a = round(f1_score(test_target, pred_label, average=metrics,zero_division=0),4)*100
acc_table[0,4] = a
print("Overall: "+str(a))

labels = labeler.transform(minerals.unique())
labels1 = labeler.inverse_transform(labels)

for n,i in enumerate(labels1):
    # data_test2 = scaler.transform(data_test1.loc[minerals==i,:])
    data_test2 = data_test1.loc[minerals==i,:].copy()
    data_test2 = pc.transform(data_test2)
    minerals2 = labeler.transform(minerals.loc[minerals == i])
    pred_label2 = model.predict(data_test2)
    a = round(f1_score(minerals2, pred_label2, average=metrics,zero_division=0),4)*100
    acc_table[n+1,4] = a
    print(labels1[n]+": "+str(round(a,2)))

data_mismatch = data_test.loc[test_target != pred_label,:]
minerals_mismatch = minerals.loc[test_target != pred_label]
data_mismatch = pd.concat([data_mismatch,minerals_mismatch],axis=1)
data_mismatch

C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but PCA was fitted without feature names
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but PCA was fitted without feature names
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but PCA was fitted without feature names
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but PCA was fitted without feature names
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but PCA was fitted without feature names
  warnings.warn(
C:\Users\naik3\Documents\pytho

Overall: 96.86
Ol: 100.0
Ms: 91.67
Grt: 80.5
Amp: 100.0
Px: 96.45
Spl: 100.0
Fsp: 100.0
Bt: 99.65
Ttn: 100.0
Ap: 100.0


C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but PCA was fitted without feature names
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but PCA was fitted without feature names
  warnings.warn(


,SiO2,MgO,FeO,TiO2,Al2O3,MnO,CaO,Na2O,K2O,P2O5,...,Sm2O3,Gd2O3,Dy2O3,Er2O3,Yb2O3,V2O3,ZrO2,CuO,CO2,Mineral
4,64.587033,10.997559,0.0,0.0,16.119588,0.0,0.0,0.0,8.295821,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Ms
5,64.633747,10.890729,0.0,0.0,16.200252,0.0,0.0,0.0,8.275272,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Ms
66,51.715743,14.21163,14.840854,0.0,9.981523,0.0,9.25025,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
68,50.033745,22.631168,7.420503,0.0,7.967547,0.0,11.947036,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
702,42.575329,31.681606,7.248057,0.0,13.297111,0.0,5.197897,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Grt
703,42.994219,30.6739,7.044727,0.0,13.331604,0.0,5.95555,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Grt
704,42.647196,32.853384,6.079312,0.0,13.413531,0.0,5.006577,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Grt
706,42.199877,32.05728,7.559745,0.0,13.445368,0.0,4.73773,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Grt
707,44.317436,34.838357,6.451083,0.0,14.393125,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Grt
709,42.726839,32.862711,5.700509,0.0,13.169931,0.0,5.54001,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Grt


In [13]:
l=['Overall']
l.extend(list(minerals.unique()))
acc_table = pd.DataFrame(acc_table,
                          columns=["C1", "C2", "C3", "C4", "C5"],index=l)
acc_table.to_excel("F1-score Accuracy report for RF_test.xlsx")
